In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import os
import glob
from typing import Dict, Any, Optional
import plotly.graph_objects as go



# Perfiles de Temperature

In [ ]:
def extract_temperature_at_max_vp_from_folders(
    folder_paths,
    vp_column='Vertical Position m',
    temp_column='Temp °C',
    recursive=False
):
    """
    Lee todos los archivos CSV de una lista de folders y extrae:
    - valor de temperatura en la mayor Vertical Position m
    - valor correspondiente de Vertical Position m
    - ID = nombre del archivo sin extensión

    Parameters
    ----------
    folder_paths : list of str
        Lista de rutas de carpetas que contienen archivos .csv
    vp_column : str
        Nombre de la columna de posición vertical
    temp_column : str
        Nombre de la columna de temperatura
    recursive : bool
        Si True, busca también en subcarpetas

    Returns
    -------
    pd.DataFrame
        Tabla con ID, temperature_at_max_vp_C, max_vp_m y file_path
    """
    results = []

    for folder in folder_paths:
        pattern = os.path.join(folder, '**', '*.csv') if recursive else os.path.join(folder, '*.csv')
        csv_files = glob.glob(pattern, recursive=recursive)

        for file_path in csv_files:
            try:
                df = pd.read_csv(file_path)

                if vp_column not in df.columns or temp_column not in df.columns:
                    print(f'[WARNING] Columnas no encontradas en: {file_path}')
                    continue

                # Convertir a numérico por si vienen strings
                df[vp_column] = pd.to_numeric(df[vp_column], errors='coerce')
                df[temp_column] = pd.to_numeric(df[temp_column], errors='coerce')

                # Quitar filas con NaN en columnas clave
                df_valid = df.dropna(subset=[vp_column, temp_column]).copy()

                if df_valid.empty:
                    print(f'[WARNING] Sin datos válidos en: {file_path}')
                    continue

                # Índice de la mayor Vertical Position m
                idx_max_vp = df_valid[vp_column].idxmax()

                max_vp = df_valid.loc[idx_max_vp, vp_column]
                temp_at_max_vp = df_valid.loc[idx_max_vp, temp_column]

                file_name = os.path.basename(file_path)
                file_id = os.path.splitext(file_name)[0]

                results.append({
                    'ID': file_id,
                    'temperature_at_max_vp_C': temp_at_max_vp,
                    'max_vp_m': max_vp,
                    'file_path': file_path
                })

            except Exception as e:
                print(f'[ERROR] No se pudo procesar {file_path}: {e}')

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df = results_df.sort_values(by='temperature_at_max_vp_C', ascending=True).reset_index(drop=True)
        results_df['temperature_at_max_vp_C'] = results_df['temperature_at_max_vp_C'].round(2)
        results_df['max_vp_m'] = results_df['max_vp_m'].round(3)

    return results_df


# =========================
# EJEMPLO DE USO
# =========================

folder_paths = [
    r'C:\Users\Mariana\Documents\freshwater_lens\data\raw\raw_bw3',
    r'C:\Users\Mariana\Documents\freshwater_lens\data\raw\raw_lrs69',
    r'C:\Users\Mariana\Documents\freshwater_lens\data\raw\raw_lrs70'
]

df_temperaturas = extract_temperature_at_max_vp_from_folders(
    folder_paths=folder_paths,
    vp_column='Vertical Position m',
    temp_column='Temp °C',
    recursive=False
)

print(df_temperaturas)

# Guardar a CSV si quieres
# df_temperaturas.to_csv('temperature_at_max_vp_summary.csv', index=False)